# 10 — State Dynamics: Registered Tests

MallWorld state-dynamics redesign (2026-10)

---

**Registration.** `docs/STATE_DYNAMICS_DESIGN_2026-10.md` was committed as registered (`6be44a3b`) before this notebook was written. The predictions, samples, models, margins and verdict rules below are copied from it and are not changed here. Where an analysis could not be run as registered, the deviation is stated where it occurs.

**Measures** (notebook 09, unchanged):
- **Q:** the place's quality, which is the scene state without atmosphere.
- **E:** the experiencer's evaluation, the coded affect.
- **Perceived benign or hostile:** an entity's demeanor as the experiencer perceived it.

**Predictions.** H1–H4 form a Holm family, one-sided at α = 0.05.

| # | Question | Prediction | Margin of no effect |
|---|---|---|---|
| H1 | Does the person lead the place, or the place the person? | E(t) → Q(t+1) exceeds Q(t) → E(t+1) | ±0.05 SD |
| H2 | Does a person move toward what he or she treats as good? | On purposeful moves, a more positive orientation (feelings running with the place's quality) goes with better destinations | ±0.05 SD |
| H3 | Is a presence the experiencer perceives as benign felt? | E is higher where a perceived-benign being is present, within the same dream | ±0.10 |
| H4 | Is what a person moves toward consistent across his or her dreams? | ICC1 of the direction of approach, by author, > 0 | ±0.05 |

**Secondary:** H5 (common structure, personal state; descriptive) and H6 (height, conditional on state). **Bias check:** B1 (contagion).

**Verdict rule.**
- **Hit:** the estimate is in the predicted direction with Holm p < 0.05.
- **Miss:** the estimate is in the opposite direction with a 95% CI excluding zero, or the 90% CI lies within the margin.
- **Underdetermined:** otherwise.

In [1]:
import sys, json, warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import chi2_contingency, spearmanr, fisher_exact
from statsmodels.stats.multitest import multipletests

MW_ROOT = Path.cwd().parent
sys.path.insert(0, str(MW_ROOT / "scripts"))
import mallworld_dataset as mw  # verified loader: schema-checked fields, populations, cluster keys

OUTPUT_DIR = MW_ROOT / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

posts = mw.load_posts()
primary = posts[posts["primary"]].copy()
print(f"Posts: {len(posts):,} | primary dream reports: {len(primary):,} from {primary['author'].nunique():,} authors")

Posts: 3,732 | primary dream reports: 1,918 from 1,303 authors


In [2]:
import re
import statsmodels.formula.api as smf
from scipy.stats import norm

L = mw.load_locations()
CON = mw.load_connections()
E_ent = mw.load_entities()

# Scoring: identical to notebook 09 (pre-specified there)
SCENE = {
    "atmosphere": {"welcoming": 1, "peaceful": 1, "neutral": 0, "uncomfortable": -1, "oppressive": -1,
                   "threatening": -1, "chaotic": -1, "eerie": -1, "wrong": -1},
    "light": {"bright_natural": 1, "bright_artificial": 0, "dim": -1, "flickering": -1, "dark": -1, "absent": -1},
    "light_temperature": {"warm_golden": 1, "neutral": 0, "cold_white": -1},
    "cleanliness": {"clean": 1, "normal": 0, "dirty": -1, "filthy": -1, "disgusting": -1},
    "state": {"new": 1, "maintained": 1, "worn": 0, "decaying": -1, "abandoned": -1, "destroyed": -1},
    "crowd_behavior": {"socializing": 1, "waiting": 0, "wandering": 0, "panic": -1, "mob": -1, "zombie_like": -1},
    "water_presence": {"pool_clean": 1, "fountain": 1, "ocean_calm": 1, "river": 0, "rain": 0, "pool_dirty": -1,
                       "flood": -1, "leak": -1, "ocean_turbulent": -1, "tsunami": -1},
}
EXPER = {
    "affective_response": {"delight": 1, "comfort": 1, "curiosity": 0, "indifference": 0, "boredom": 0,
                           "confusion": -1, "anxiety": -1, "disgust": -1, "horror": -1},
    "somatic_response": {"comfort": 1, "nausea": -1, "dizziness": -1, "paralysis": -1, "heaviness": -1,
                         "headache": -1, "sleepiness": -1},
}
for table in (SCENE, EXPER):
    for col, mapping in table.items():
        mw.check_values(col, mapping)
for col, mapping in {**SCENE, **EXPER}.items():
    L["s_" + col] = L[col].map(mapping).astype(float)
scene_cols = ["s_" + c for c in SCENE]
L["Q"] = L[[c for c in scene_cols if c != "s_atmosphere"]].mean(axis=1)
L["Q_full"] = L[scene_cols].mean(axis=1)                                                  # with atmosphere
L["Q_nolt"] = L[[c for c in scene_cols if c not in ("s_atmosphere", "s_light_temperature")]].mean(axis=1)
L["E"] = L[["s_" + c for c in EXPER]].mean(axis=1)

# Beings, as perceived
DEM = {"helpful": "benign", "friendly": "benign", "hostile": "hostile", "threatening": "hostile", "unfriendly": "hostile"}
mw.check_values("demeanor", DEM)
cls = E_ent.assign(c=E_ent.demeanor.map(DEM)).dropna(subset=["c"]).groupby(["post_id", "location_id"]).c.agg(set)
L = L.join(cls.rename("beings"), on=["post_id", "location_id"])
L["benign"] = L.beings.apply(lambda s: isinstance(s, set) and "benign" in s).astype(int)
L["hostile"] = L.beings.apply(lambda s: isinstance(s, set) and "hostile" in s).astype(int)

# Dream-level strata (patterns from notebook 09, M0)
prim = posts[posts.primary].copy()
text = prim.post_id.map(mw.raw_text)
ONSET = re.compile(r"\b(since i was (a )?(kid|child|little|young|teen|teenager|\d+)|as a (kid|child)|when i was (a )?(kid|child|little|young)"
                   r"|for (many |several |\d+ )?years|my whole life|all my life|childhood)\b", re.I)
prim["onset"] = text.str.contains(ONSET).astype(int)
prim["vocab"] = text.str.contains(re.compile(r"\bmall ?world\b", re.I)).astype(int)
prim["date"] = pd.to_datetime(prim["date"], utc=True)
prim["first_report"] = (prim.groupby("author").date.rank(method="first") == 1).astype(int)
prim["early_era"] = (prim.year <= 2023).astype(int)
strata = prim.set_index("post_id")[["onset", "vocab", "first_report", "early_era"]]

# Transitions: explicit connections joined on the full key
def build_T(qcol="Q", ecol="E", drop_self_loops=True):
    k = L[["post_id", "location_id", qcol, ecol]]
    T = (CON.merge(k.rename(columns={"location_id": "from_location_id", qcol: "qf", ecol: "ef"}), on=["post_id", "from_location_id"], how="left")
          .merge(k.rename(columns={"location_id": "to_location_id", qcol: "qt", ecol: "et"}), on=["post_id", "to_location_id"], how="left"))
    if drop_self_loops:
        T = T[T.from_location_id != T.to_location_id]
    return T.reset_index(drop=True)

# List-order pairs (fallback; no movement mode)
def build_LO(qcol="Q", ecol="E"):
    Ls = L.sort_values(["post_id", "list_index"])
    nx = Ls.groupby("post_id")[["location_id", qcol, ecol]].shift(-1)
    out = pd.DataFrame({"post_id": Ls.post_id.values, "author": Ls.author.values, "log_words": Ls.log_words.values,
                        "from_location_id": Ls.location_id.values, "to_location_id": nx.location_id.values,
                        "qf": Ls[qcol].values, "qt": nx[qcol].values, "ef": Ls[ecol].values, "et": nx[ecol].values})
    out["transit_mode"] = "list_order"
    return out.dropna(subset=["to_location_id"]).reset_index(drop=True)

T0 = build_T()
self_loops = int((CON.from_location_id == CON.to_location_id).sum())
print(f"Connections: {len(CON):,}; connections from a location to itself, excluded as not a movement: {self_loops}")

# Helpers
def cl_ols(formula, d, cluster="post_id"):
    m = smf.ols(formula, data=d)
    groups = pd.factorize(d.loc[m.data.row_labels, cluster])[0]
    return m.fit(cov_type="cluster", cov_kwds={"groups": groups})

def lstsq(X, y):
    return np.linalg.lstsq(X, y, rcond=None)[0]

def cluster_boot(d, stat, B=2000, seed=0, cluster="post_id"):
    idx = list(d.groupby(cluster).indices.values())
    rng = np.random.default_rng(seed)
    out = []
    for _ in range(B):
        pick = rng.integers(0, len(idx), len(idx))
        out.append(stat(d.iloc[np.concatenate([idx[i] for i in pick])]))
    return np.array(out)

def icc1(df, group="author", y="score"):
    g = df.groupby(group)[y]
    k = g.size(); n = len(df); a = len(k)
    grand = df[y].mean()
    msb = (k * (g.mean() - grand) ** 2).sum() / (a - 1)
    msw = ((df[y] - g.transform("mean")) ** 2).sum() / (n - a)
    k0 = (n - (k ** 2).sum() / n) / (a - 1)
    return (msb - msw) / (msb + (k0 - 1) * msw)

def icc1_np(y, g):
    a = g.max() + 1; n = len(y)
    k = np.bincount(g, minlength=a); means = np.bincount(g, weights=y, minlength=a) / k
    msb = (k * (means - y.mean()) ** 2).sum() / (a - 1)
    msw = ((y - means[g]) ** 2).sum() / (n - a)
    k0 = (n - (k ** 2).sum() / n) / (a - 1)
    return (msb - msw) / (msb + (k0 - 1) * msw)

def author_boot_icc(df, y, B=2000, seed=0):
    rng = np.random.default_rng(seed)
    codes, uniq = pd.factorize(df.author)
    idx = [np.flatnonzero(codes == i) for i in range(len(uniq))]
    yv = df[y].to_numpy(float)
    out = []
    for _ in range(B):
        pick = rng.integers(0, len(idx), len(idx))
        rows = np.concatenate([idx[i] for i in pick])
        labels = np.repeat(np.arange(len(pick)), [len(idx[i]) for i in pick])
        out.append(icc1_np(yv[rows], labels))
    return np.array(out)

def verdict(est, p_holm, ci95, ci90, margin):
    if est > 0 and p_holm < 0.05:
        return "HIT"
    if (est < 0 and ci95[1] < 0) or (ci90[0] > -margin and ci90[1] < margin):
        return "MISS"
    return "UNDERDETERMINED"

Connections: 4,590; connections from a location to itself, excluded as not a movement: 259


/tmp/ipykernel_2804/3427477290.py:51: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  prim["onset"] = text.str.contains(ONSET).astype(int)


## H1 — Does the person lead the place, or the place the person?

On transitions with Q and E at both ends, Q and E are standardised over all their values in the sample. Model A: Q(t+1) ~ Q(t) + E(t) + log_words. Model B: E(t+1) ~ E(t) + Q(t) + log_words. D = the coefficient of E(t) in A minus the coefficient of Q(t) in B. Dream bootstrap, 2,000 resamples.

In [3]:
def h1_coefs(d):
    q = np.concatenate([d.qf.values, d.qt.values]); e = np.concatenate([d.ef.values, d.et.values])
    Qf, Qt = (d.qf.values - q.mean()) / q.std(ddof=1), (d.qt.values - q.mean()) / q.std(ddof=1)
    Ef, Et = (d.ef.values - e.mean()) / e.std(ddof=1), (d.et.values - e.mean()) / e.std(ddof=1)
    one, lw = np.ones(len(d)), d.log_words.values
    bA = lstsq(np.column_stack([one, Qf, Ef, lw]), Qt)
    bB = lstsq(np.column_stack([one, Ef, Qf, lw]), Et)
    return bA[2], bB[2], bA[2] - bB[2]

def run_h1(T, B=2000, label=""):
    d = T.dropna(subset=["qf", "qt", "ef", "et", "log_words"]).reset_index(drop=True)
    bE, bS, D = h1_coefs(d)
    boot = cluster_boot(d, lambda x: h1_coefs(x)[2], B=B)
    p = (1 + (boot <= 0).sum()) / (1 + B)
    return dict(label=label, n=len(d), dreams=d.post_id.nunique(), E_to_Q=bE, Q_to_E=bS, D=D, p=p,
                ci95=tuple(np.percentile(boot, [2.5, 97.5])), ci90=tuple(np.percentile(boot, [5, 95])))

h1 = run_h1(T0, label="primary")
d1 = T0.dropna(subset=["qf", "qt", "ef", "et"])
print(f"H1 sample: {h1['n']} transitions in {h1['dreams']} dreams")
print(f"  person -> place  (E(t) on Q(t+1), given Q(t)): {h1['E_to_Q']:+.3f} SD")
print(f"  place -> person  (Q(t) on E(t+1), given E(t)): {h1['Q_to_E']:+.3f} SD")
print(f"  D = {h1['D']:+.3f}; 95% CI {h1['ci95'][0]:+.3f} to {h1['ci95'][1]:+.3f}; 90% CI {h1['ci90'][0]:+.3f} to {h1['ci90'][1]:+.3f}; one-sided p = {h1['p']:.4f}")

# The same two models with clustered standard errors, for transparency
q = pd.concat([d1.qf, d1.qt]); e = pd.concat([d1.ef, d1.et])
dz = d1.assign(Qf=(d1.qf - q.mean()) / q.std(), Qt=(d1.qt - q.mean()) / q.std(), Ef=(d1.ef - e.mean()) / e.std(), Et=(d1.et - e.mean()) / e.std())
mA = cl_ols("Qt ~ Qf + Ef + log_words", dz); mB = cl_ols("Et ~ Ef + Qf + log_words", dz)
print("\nModel A (place at t+1):"); print(mA.summary2().tables[1][["Coef.", "Std.Err.", "P>|z|"]].round(3).to_string())
print("\nModel B (person at t+1):"); print(mB.summary2().tables[1][["Coef.", "Std.Err.", "P>|z|"]].round(3).to_string())

H1 sample: 711 transitions in 350 dreams
  person -> place  (E(t) on Q(t+1), given Q(t)): +0.117 SD
  place -> person  (Q(t) on E(t+1), given E(t)): -0.003 SD
  D = +0.121; 95% CI +0.020 to +0.213; 90% CI +0.036 to +0.195; one-sided p = 0.0120

Model A (place at t+1):
           Coef.  Std.Err.  P>|z|
Intercept -0.322     0.232  0.165
Qf         0.460     0.041  0.000
Ef         0.117     0.035  0.001
log_words  0.048     0.038  0.201

Model B (person at t+1):
           Coef.  Std.Err.  P>|z|
Intercept -0.117     0.239  0.624
Ef         0.455     0.041  0.000
Qf        -0.003     0.035  0.922
log_words  0.007     0.039  0.854


## H2 — Does a person move toward what he or she treats as good?

Orientation is the mean of E·Q over the dream's other locations with both measures, with the transition's origin and destination left out; at least two are required. On purposeful moves (`directed_active`) with Q at both ends: Q(to), standardised, ~ Q(from) + orientation + log_words.

In [4]:
def orientation_table(qcol):
    b = L.dropna(subset=[qcol, "E"])
    return {pid: g.set_index("location_id")[qcol].mul(g.set_index("location_id")["E"]) for pid, g in b.groupby("post_id")}

def add_orientation(T, qcol, minimum=2):
    tab = orientation_table(qcol)
    def f(r):
        s = tab.get(r.post_id)
        if s is None:
            return np.nan
        s = s.drop([r.from_location_id, r.to_location_id], errors="ignore")
        return s.mean() if len(s) >= minimum else np.nan
    return T.assign(orient=T.apply(f, axis=1))

def run_h2(T, qcol="Q", purposeful=True, label=""):
    d = T[T.transit_mode.eq("directed_active")] if purposeful else T
    d = add_orientation(d.dropna(subset=["qf", "qt", "log_words"]), qcol).dropna(subset=["orient"]).copy()
    d["qt_z"] = (d.qt - d.qt.mean()) / d.qt.std()
    m = cl_ols("qt_z ~ qf + orient + log_words", d)
    est, se = m.params["orient"], m.bse["orient"]
    return dict(label=label, n=len(d), dreams=d.post_id.nunique(), est=est, se=se, p=1 - norm.cdf(est / se),
                ci95=(est - 1.96 * se, est + 1.96 * se), ci90=(est - 1.645 * se, est + 1.645 * se),
                share_neg_orient=float((d.orient < 0).mean()), d=d)

h2 = run_h2(T0, label="primary")
print(f"H2 sample: {h2['n']} purposeful moves in {h2['dreams']} dreams with an estimable orientation")
print(f"  orientation: mean {h2['d'].orient.mean():+.3f}; share of moves with negative orientation {h2['share_neg_orient']:.1%}; "
      f"with orientation <= -0.25: {(h2['d'].orient <= -0.25).mean():.1%}")
print(f"  coefficient of orientation on destination quality: {h2['est']:+.3f} SD per unit "
      f"(95% CI {h2['ci95'][0]:+.3f} to {h2['ci95'][1]:+.3f}; 90% CI {h2['ci90'][0]:+.3f} to {h2['ci90'][1]:+.3f}); one-sided p = {h2['p']:.4f}")

H2 sample: 425 purposeful moves in 163 dreams with an estimable orientation
  orientation: mean +0.107; share of moves with negative orientation 32.0%; with orientation <= -0.25: 22.1%
  coefficient of orientation on destination quality: -0.245 SD per unit (95% CI -0.436 to -0.055; 90% CI -0.405 to -0.085); one-sided p = 0.9942


## H3 — Is a presence the experiencer perceives as benign felt?

Locations with E in dreams that have at least one location with and one without a perceived-benign being (both with E). Model: E ~ benign + hostile + dream fixed effects, with standard errors clustered by dream.

In [5]:
def run_h3(Lsrc, hostile_control=True, label=""):
    b = Lsrc.dropna(subset=["E"])
    g = b.groupby("post_id").benign.agg(["sum", "size"])
    keep = g[(g["sum"] >= 1) & (g["sum"] < g["size"])].index
    d = b[b.post_id.isin(keep)]
    m = cl_ols("E ~ benign" + (" + hostile" if hostile_control else "") + " + C(post_id)", d)
    est, se = m.params["benign"], m.bse["benign"]
    return dict(label=label, n=len(d), dreams=d.post_id.nunique(), est=est, se=se, p=1 - norm.cdf(est / se),
                ci95=(est - 1.96 * se, est + 1.96 * se), ci90=(est - 1.645 * se, est + 1.645 * se),
                hostile=(m.params.get("hostile", np.nan), m.bse.get("hostile", np.nan)), d=d)

h3 = run_h3(L, label="primary")
print(f"H3 sample: {h3['n']} locations in {h3['dreams']} dreams; with a perceived-benign being: {int(h3['d'].benign.sum())}")
print(f"  mean E with / without a perceived-benign being: {h3['d'][h3['d'].benign == 1].E.mean():+.3f} / {h3['d'][h3['d'].benign == 0].E.mean():+.3f}")
print(f"  within-dream effect of a perceived-benign presence on E: {h3['est']:+.3f} "
      f"(95% CI {h3['ci95'][0]:+.3f} to {h3['ci95'][1]:+.3f}; 90% CI {h3['ci90'][0]:+.3f} to {h3['ci90'][1]:+.3f}); one-sided p = {h3['p']:.4f}")
print(f"  (perceived-hostile presence, same model: {h3['hostile'][0]:+.3f}, SE {h3['hostile'][1]:.3f})")

H3 sample: 864 locations in 175 dreams; with a perceived-benign being: 225
  mean E with / without a perceived-benign being: +0.082 / -0.329
  within-dream effect of a perceived-benign presence on E: +0.353 (95% CI +0.221 to +0.485; 90% CI +0.242 to +0.464); one-sided p = 0.0000
  (perceived-hostile presence, same model: -0.445, SE 0.080)


## H4 — Is what a person moves toward consistent across his or her dreams?

For purposeful moves with Q at both ends, the direction of approach is the residual of Q(to) on Q(from) and log_words, averaged per dream. ICC1 of the dream means by author, over authors with at least two such dreams. One-sided p from 5,000 permutations of author labels; 90% CI from 2,000 author bootstrap resamples.

In [6]:
def approach_dreams(T, purposeful=True):
    d = T[T.transit_mode.eq("directed_active")] if purposeful else T
    d = d.dropna(subset=["qf", "qt", "log_words"]).copy()
    X = np.column_stack([np.ones(len(d)), d.qf, d.log_words])
    d["resid"] = d.qt - X @ lstsq(X, d.qt.values)
    dm = d.groupby("post_id").agg(score=("resid", "mean"), author=("author", "first"))
    return dm[dm.author.map(dm.author.value_counts()) >= 2].copy()

def run_h4(T, purposeful=True, label="", B=2000, P=5000):
    dm = approach_dreams(T, purposeful)
    obs = icc1(dm)
    codes = pd.factorize(dm.author)[0]; yv = dm.score.to_numpy(float)
    assert abs(icc1_np(yv, codes) - obs) < 1e-12
    rng = np.random.default_rng(0)
    perm = np.array([icc1_np(yv, rng.permutation(codes)) for _ in range(P)])
    boot = author_boot_icc(dm, "score", B=B)
    return dict(label=label, n=len(dm), authors=dm.author.nunique(), est=obs, p=(1 + (perm >= obs).sum()) / (1 + P),
                null_mean=perm.mean(), ci95=tuple(np.percentile(boot, [2.5, 97.5])), ci90=tuple(np.percentile(boot, [5, 95])))

h4 = run_h4(T0, label="primary")
print(f"H4 sample: {h4['n']} dreams from {h4['authors']} authors")
print(f"  ICC1 = {h4['est']:+.3f} (permutation null mean {h4['null_mean']:+.3f}); one-sided p = {h4['p']:.4f}; "
      f"95% CI {h4['ci95'][0]:+.3f} to {h4['ci95'][1]:+.3f}; 90% CI {h4['ci90'][0]:+.3f} to {h4['ci90'][1]:+.3f}")

H4 sample: 116 dreams from 38 authors
  ICC1 = +0.148 (permutation null mean -0.000); one-sided p = 0.0628; 95% CI -0.056 to +0.353; 90% CI -0.030 to +0.315


## Verdicts (Holm family H1–H4)

In [7]:
fam = pd.Series({"H1": h1["p"], "H2": h2["p"], "H3": h3["p"], "H4": h4["p"]})
holm = pd.Series(multipletests(fam.values, method="holm")[1], index=fam.index)
est = {"H1": h1["D"], "H2": h2["est"], "H3": h3["est"], "H4": h4["est"]}
margin = {"H1": 0.05, "H2": 0.05, "H3": 0.10, "H4": 0.05}
res = {"H1": h1, "H2": h2, "H3": h3, "H4": h4}
rows = []
for k in fam.index:
    r = res[k]
    rows.append({"test": k, "estimate": est[k], "95% CI": f"{r['ci95'][0]:+.3f} to {r['ci95'][1]:+.3f}",
                 "90% CI": f"{r['ci90'][0]:+.3f} to {r['ci90'][1]:+.3f}", "margin": f"±{margin[k]}",
                 "one-sided p": fam[k], "Holm p": holm[k], "verdict": verdict(est[k], holm[k], r["ci95"], r["ci90"], margin[k])})
verdicts = pd.DataFrame(rows).set_index("test")
with pd.option_context("display.float_format", "{:.4f}".format):
    print(verdicts.to_string())

      estimate            95% CI            90% CI margin  one-sided p  Holm p          verdict
test                                                                                           
H1      0.1208  +0.020 to +0.213  +0.036 to +0.195  ±0.05       0.0120  0.0360              HIT
H2     -0.2453  -0.436 to -0.055  -0.405 to -0.085  ±0.05       0.9942  0.9942             MISS
H3      0.3531  +0.221 to +0.485  +0.242 to +0.464   ±0.1       0.0000  0.0000              HIT
H4      0.1477  -0.056 to +0.353  -0.030 to +0.315  ±0.05       0.0628  0.1256  UNDERDETERMINED


## Deviation check: connections from a location to itself

The registration defined transitions as explicit connections with both ends coded. 259 connections lead from a location to itself. They are not movements, so they were excluded above. That exclusion was not written in the registration, so the primary tests are re-run here with them included.

In [8]:
Tall = build_T(drop_self_loops=False)
dev = pd.DataFrame([
    {"test": "H1", **{k: v for k, v in run_h1(Tall, B=1000, label="with self-loops").items() if k in ("n", "D", "p", "ci95")}},
    {"test": "H2", **{k: v for k, v in run_h2(Tall, label="with self-loops").items() if k in ("n", "est", "p", "ci95")}},
    {"test": "H4", **{k: v for k, v in run_h4(Tall, label="with self-loops", B=1000, P=2000).items() if k in ("n", "est", "p", "ci95")}},
])
with pd.option_context("display.float_format", "{:.4f}".format):
    print(dev.to_string(index=False))

test   n      D      p                                        ci95     est
  H1 801 0.1072 0.0100   (0.02409524719727134, 0.1854816895266926)     NaN
  H2 438    NaN 0.9919 (-0.4150544623639808, -0.04236272362538443) -0.2287
  H4 126    NaN 0.0920  (-0.09517152426278389, 0.3306931609628492)  0.1215


## H2 — post hoc diagnosis of the miss (exploratory; the verdict stands)

The orientation index was meant to separate people whose feelings run with the place's quality from people who treat darkness as good. Three checks, written after seeing the result:
1. what makes an index negative;
2. whether the index is confounded with the dream's overall level (mean Q and mean E at the same other locations);
3. a rank-correlation index over at least three other locations.

In [9]:
b = L.dropna(subset=["Q", "E"])
negp = b[(b.Q * b.E) < 0]
print(f"Locations with a negative E·Q product: {len(negp)}; distress in a better-than-neutral place (E < 0, Q > 0): "
      f"{mw.fmt_pct(int(((negp.E < 0) & (negp.Q > 0)).sum()), len(negp))}; comfort in a poor place (E > 0, Q < 0): "
      f"{mw.fmt_pct(int(((negp.E > 0) & (negp.Q < 0)).sum()), len(negp))}")

def other_means(T, qcol="Q"):
    bb = L.dropna(subset=[qcol, "E"])
    tabs = {pid: g.set_index("location_id")[[qcol, "E"]] for pid, g in bb.groupby("post_id")}
    def f(r):
        t = tabs.get(r.post_id)
        if t is None:
            return pd.Series({"oq": np.nan, "oe": np.nan, "orho": np.nan})
        t = t.drop([r.from_location_id, r.to_location_id], errors="ignore")
        rho = spearmanr(t[qcol], t["E"]).correlation if len(t) >= 3 and t[qcol].nunique() > 1 and t["E"].nunique() > 1 else np.nan
        return pd.Series({"oq": t[qcol].mean(), "oe": t["E"].mean(), "orho": rho})
    return T.join(T.apply(f, axis=1))

dd = other_means(h2["d"])
m_lvl = cl_ols("qt_z ~ qf + orient + oq + oe + log_words", dd)
print("\nRegistered model plus the dream's mean Q and mean E at the same other locations:")
print(m_lvl.summary2().tables[1][["Coef.", "Std.Err.", "P>|z|"]].round(3).to_string())

d_rho = other_means(T0[T0.transit_mode.eq("directed_active")].dropna(subset=["qf", "qt", "log_words"])).dropna(subset=["orho"]).copy()
d_rho["qt_z"] = (d_rho.qt - d_rho.qt.mean()) / d_rho.qt.std()
m_rho = cl_ols("qt_z ~ qf + orho + oq + oe + log_words", d_rho)
print(f"\nRank-correlation orientation (>= 3 other locations): {len(d_rho)} moves in {d_rho.post_id.nunique()} dreams; "
      f"share with rho < 0: {(d_rho.orho < 0).mean():.1%}")
print(m_rho.summary2().tables[1][["Coef.", "Std.Err.", "P>|z|"]].round(3).to_string())

Locations with a negative E·Q product: 482; distress in a better-than-neutral place (E < 0, Q > 0): 447/482 = 92.7% [95% CI 90.1–94.7]; comfort in a poor place (E > 0, Q < 0): 35/482 = 7.3% [95% CI 5.3–9.9]



Registered model plus the dream's mean Q and mean E at the same other locations:
           Coef.  Std.Err.  P>|z|
Intercept -0.223     0.358  0.532
qf         0.533     0.078  0.000
orient    -0.235     0.149  0.117
oq         0.157     0.151  0.297
oe         0.135     0.127  0.288
log_words -0.001     0.053  0.983



Rank-correlation orientation (>= 3 other locations): 257 moves in 81 dreams; share with rho < 0: 16.7%
           Coef.  Std.Err.  P>|z|
Intercept  0.446     0.642  0.487
qf         0.484     0.116  0.000
orho      -0.070     0.110  0.524
oq         0.230     0.200  0.250
oe        -0.117     0.158  0.461
log_words -0.112     0.096  0.244


## Sensitivity analyses (registered; they do not change the verdicts)

- **Measure variants:** Q with atmosphere added (`Q_full`); Q without light temperature (`Q_nolt`).
- **Unit variant:** list-order pairs in place of explicit transitions. List-order pairs carry no movement mode. For H2 and H4 they therefore use all consecutive pairs, not only purposeful moves; this is a deviation forced by the data.
- **Dream fixed effects:** for H1. For H2 they are not estimable, because orientation is a property of the dream: within a dream it varies only through which two locations are left out.
- **Strata:** onset mention and community vocabulary (H1, H2, H3).
- **H3:** also without the hostile control.

Bootstraps use 1,000 resamples here.

In [10]:
def h1_fe(T, B=1000):
    d = T.dropna(subset=["qf", "qt", "ef", "et", "log_words"]).reset_index(drop=True)
    def stat(x):
        q = np.concatenate([x.qf.values, x.qt.values]); e = np.concatenate([x.ef.values, x.et.values])
        z = pd.DataFrame({"Qf": (x.qf.values - q.mean()) / q.std(ddof=1), "Qt": (x.qt.values - q.mean()) / q.std(ddof=1),
                          "Ef": (x.ef.values - e.mean()) / e.std(ddof=1), "Et": (x.et.values - e.mean()) / e.std(ddof=1),
                          "g": x.post_id.values})
        z = z.groupby("g").filter(lambda g: len(g) >= 2)
        w = z[["Qf", "Qt", "Ef", "Et"]] - z.groupby("g")[["Qf", "Qt", "Ef", "Et"]].transform("mean")
        bA = lstsq(w[["Qf", "Ef"]].values, w.Qt.values); bB = lstsq(w[["Ef", "Qf"]].values, w.Et.values)
        return bA[1] - bB[1]
    boot = cluster_boot(d, stat, B=B)
    return dict(label="H1 dream fixed effects", D=stat(d), ci95=tuple(np.percentile(boot, [2.5, 97.5])),
                p=(1 + (boot <= 0).sum()) / (1 + B), n=int(d.groupby("post_id").size().ge(2).sum()))

rows = []
def add(test, r, key="D"):
    rows.append({"test": test, "variant": r["label"], "n": r.get("n"), "estimate": r[key],
                 "95% CI": f"{r['ci95'][0]:+.3f} to {r['ci95'][1]:+.3f}", "one-sided p": r["p"]})

for qcol in ("Q_full", "Q_nolt"):
    add("H1", run_h1(build_T(qcol), B=1000, label=qcol))
add("H1", run_h1(build_LO(), B=1000, label="list-order pairs"))
r = h1_fe(T0); rows.append({"test": "H1", "variant": r["label"], "n": r["n"], "estimate": r["D"],
                            "95% CI": f"{r['ci95'][0]:+.3f} to {r['ci95'][1]:+.3f}", "one-sided p": r["p"]})
for s in ("onset", "vocab"):
    for v in (1, 0):
        Ts = T0[T0.post_id.map(strata[s]) == v]
        add("H1", run_h1(Ts, B=1000, label=f"{s} = {v}"))

for qcol in ("Q_full", "Q_nolt"):
    add("H2", run_h2(build_T(qcol), qcol=qcol, label=qcol), key="est")
add("H2", run_h2(build_LO(), purposeful=False, label="list-order pairs (all)"), key="est")
for s in ("onset", "vocab"):
    for v in (1, 0):
        try:
            add("H2", run_h2(T0[T0.post_id.map(strata[s]) == v], label=f"{s} = {v}"), key="est")
        except Exception as ex:
            rows.append({"test": "H2", "variant": f"{s} = {v}", "n": None, "estimate": np.nan, "95% CI": f"not estimable ({type(ex).__name__})", "one-sided p": np.nan})

add("H3", run_h3(L, hostile_control=False, label="without hostile control"), key="est")
for s in ("onset", "vocab"):
    for v in (1, 0):
        add("H3", run_h3(L[L.post_id.map(strata[s]) == v], label=f"{s} = {v}"), key="est")

for qcol in ("Q_full", "Q_nolt"):
    r = run_h4(build_T(qcol), label=qcol, B=1000, P=2000); add("H4", r, key="est")
r = run_h4(build_LO(), purposeful=False, label="list-order pairs (all)", B=1000, P=2000); add("H4", r, key="est")

sens = pd.DataFrame(rows)
with pd.option_context("display.float_format", "{:.4f}".format, "display.max_colwidth", 40):
    print(sens.to_string(index=False))

test                 variant    n  estimate           95% CI  one-sided p
  H1                  Q_full 1322    0.1016 -0.008 to +0.208       0.0350
  H1                  Q_nolt  699    0.1171 +0.025 to +0.203       0.0070
  H1        list-order pairs  865    0.0613 -0.026 to +0.138       0.0809
  H1  H1 dream fixed effects  160    0.0645 -0.122 to +0.247       0.2488
  H1               onset = 1  140    0.3606 +0.108 to +0.604       0.0030
  H1               onset = 0  571    0.0682 -0.041 to +0.171       0.1039
  H1               vocab = 1  350    0.1276 -0.033 to +0.267       0.0619
  H1               vocab = 0  361    0.1116 -0.017 to +0.227       0.0410
  H2                  Q_full  799   -0.3796 -0.576 to -0.183       0.9999
  H2                  Q_nolt  421   -0.2645 -0.454 to -0.075       0.9969
  H2  list-order pairs (all)  991   -0.1855 -0.315 to -0.056       0.9974
  H2               onset = 1  106   -0.0067 -0.397 to +0.383       0.5134
  H2               onset = 0  319   -0

## H5 (secondary, descriptive) — Common structure, personal state

ICC1 by author, over authors with at least two primary dreams, of the dream's state (mean Q and mean E) and of the dream's structure (presence of each of the ten most common location types other than "other", and of escalator, elevator and stairs connections). Author bootstrap, 2,000 resamples.

In [11]:
multi_authors = prim.author.value_counts()
multi_authors = set(multi_authors[multi_authors >= 2].index)
D = prim[prim.author.isin(multi_authors)][["post_id", "author"]].set_index("post_id")
LOC_MOTIFS = ["mall", "mall_store", "city_street", "house", "school", "hotel", "restaurant", "beach", "parking_lot", "neighborhood"]
CON_MOTIFS = ["escalator", "elevator", "stairs"]
mw.check_values("location_type", LOC_MOTIFS); mw.check_values("connection_type", CON_MOTIFS)
for m_ in LOC_MOTIFS:
    D["loc_" + m_] = D.index.isin(L.loc[L.location_type.eq(m_), "post_id"]).astype(float)
for m_ in CON_MOTIFS:
    D["con_" + m_] = D.index.isin(CON.loc[CON.connection_type.eq(m_), "post_id"]).astype(float)
D["mean_Q"] = L.groupby("post_id").Q.mean().reindex(D.index)
D["mean_E"] = L.groupby("post_id").E.mean().reindex(D.index)

rows = []
for col in ["mean_Q", "mean_E"] + ["loc_" + m_ for m_ in LOC_MOTIFS] + ["con_" + m_ for m_ in CON_MOTIFS]:
    d = D.dropna(subset=[col]).rename(columns={col: "score"})[["author", "score"]]
    d = d[d.author.map(d.author.value_counts()) >= 2]
    if d.score.nunique() < 2:
        continue
    boot = author_boot_icc(d, "score", B=2000)
    rows.append({"feature": col, "kind": "state" if col.startswith("mean") else "structure", "dreams": len(d),
                 "authors": d.author.nunique(), "prevalence/mean": d.score.mean(), "ICC1": icc1(d),
                 "95% CI": f"{np.percentile(boot, 2.5):+.3f} to {np.percentile(boot, 97.5):+.3f}"})
h5 = pd.DataFrame(rows)
with pd.option_context("display.float_format", "{:.3f}".format):
    print(h5.to_string(index=False))
print(f"\nMean ICC1, state: {h5[h5.kind == 'state'].ICC1.mean():+.3f}; structure: {h5[h5.kind == 'structure'].ICC1.mean():+.3f}")

         feature      kind  dreams  authors  prevalence/mean   ICC1           95% CI
          mean_Q     state     576      188            0.168  0.019 -0.067 to +0.107
          mean_E     state     566      184           -0.388  0.110 +0.027 to +0.190
        loc_mall structure     893      278            0.330  0.185 +0.111 to +0.263
  loc_mall_store structure     893      278            0.185  0.053 -0.017 to +0.127
 loc_city_street structure     893      278            0.170  0.087 +0.013 to +0.170
       loc_house structure     893      278            0.132  0.009 -0.059 to +0.090
      loc_school structure     893      278            0.103  0.092 +0.000 to +0.203
       loc_hotel structure     893      278            0.104  0.139 +0.031 to +0.249
  loc_restaurant structure     893      278            0.091  0.124 +0.009 to +0.248
       loc_beach structure     893      278            0.083  0.062 -0.048 to +0.179
 loc_parking_lot structure     893      278            0.071 -0.0

## H6 (secondary) — Height, conditional on state

Transitions with direction up, diagonal_up, down or diagonal_down, with E at both ends and Q at the origin. Model: E(t+1) ~ E(t) × up + Q(t) + log_words.

In [12]:
UPDOWN = {"up": 1, "diagonal_up": 1, "down": 0, "diagonal_down": 0}
mw.check_values("direction", UPDOWN)
d6 = T0[T0.direction.isin(UPDOWN)].dropna(subset=["ef", "et", "qf", "log_words"]).assign(up=lambda x: x.direction.map(UPDOWN))
m6 = cl_ols("et ~ ef * up + qf + log_words", d6)
est, se = m6.params["ef:up"], m6.bse["ef:up"]
print(f"H6 sample: {len(d6)} vertical transitions in {d6.post_id.nunique()} dreams ({int(d6.up.sum())} up, {int((1 - d6.up).sum())} down)")
print(m6.summary2().tables[1][["Coef.", "Std.Err.", "P>|z|"]].round(3).to_string())
print(f"\nInteraction E(t) x up: {est:+.3f} (95% CI {est - 1.96 * se:+.3f} to {est + 1.96 * se:+.3f}); one-sided p = {1 - norm.cdf(est / se):.4f}")

H6 sample: 226 vertical transitions in 164 dreams (121 up, 105 down)
           Coef.  Std.Err.  P>|z|
Intercept -0.501     0.325  0.123
ef         0.399     0.096  0.000
up         0.182     0.112  0.105
ef:up     -0.116     0.127  0.362
qf         0.119     0.062  0.055
log_words  0.007     0.057  0.903

Interaction E(t) x up: -0.116 (95% CI -0.364 to +0.133); one-sided p = 0.8189


## Descriptive — Perceived beings by the quality of the place

In [13]:
dq = L.dropna(subset=["Q"]).assign(sign=lambda x: np.sign(x.Q).map({-1: "Q < 0", 0: "Q = 0", 1: "Q > 0"}))
tab = dq.groupby("sign").agg(locations=("Q", "size"), perceived_benign=("benign", "mean"), perceived_hostile=("hostile", "mean"))
with pd.option_context("display.float_format", "{:.3f}".format):
    print(tab.to_string())

       locations  perceived_benign  perceived_hostile
sign                                                 
Q < 0       1071             0.032              0.158
Q = 0        737             0.061              0.122
Q > 0       2074             0.087              0.074


## B1 — Bias check: is the common structure contagion?

Dream-level presence of the 13 structural motifs. Each split's length-adjusted difference is the average marginal effect from a logistic model (motif ~ group + log_words), with a 90% dream-bootstrap CI (1,000 resamples). A motif counts as equivalent when the CI lies within ±10 percentage points.

In [14]:
def logit_fit(X, y, iters=50):
    b = np.zeros(X.shape[1])
    for _ in range(iters):
        p = 1 / (1 + np.exp(-(X @ b)))
        W = p * (1 - p)
        step = np.linalg.solve(X.T @ (X * W[:, None]) + 1e-9 * np.eye(X.shape[1]), X.T @ (y - p))
        b += step
        if np.abs(step).max() < 1e-8:
            break
    return b

def ame(X, y):
    b = logit_fit(X, y)
    X1, X0 = X.copy(), X.copy(); X1[:, 1] = 1; X0[:, 1] = 0
    return (1 / (1 + np.exp(-(X1 @ b))) - 1 / (1 + np.exp(-(X0 @ b)))).mean()

P = prim.set_index("post_id")[["log_words", "onset", "first_report", "early_era"]].copy()
for m_ in LOC_MOTIFS:
    P["loc_" + m_] = P.index.isin(L.loc[L.location_type.eq(m_), "post_id"]).astype(float)
for m_ in CON_MOTIFS:
    P["con_" + m_] = P.index.isin(CON.loc[CON.connection_type.eq(m_), "post_id"]).astype(float)
motifs = ["loc_" + m_ for m_ in LOC_MOTIFS] + ["con_" + m_ for m_ in CON_MOTIFS]
rng = np.random.default_rng(0)
boots = [rng.integers(0, len(P), len(P)) for _ in range(1000)]
rows = []
for split in ("onset", "first_report", "early_era"):
    for m_ in motifs:
        X = np.column_stack([np.ones(len(P)), P[split].values, P.log_words.values]); y = P[m_].values
        a = ame(X, y)
        bs = np.array([ame(X[i], y[i]) for i in boots])
        lo, hi = np.percentile(bs, [5, 95])
        rows.append({"split": split, "motif": m_, "rate": y.mean(), "group 1 n": int(P[split].sum()),
                     "AME (pp)": 100 * a, "90% CI (pp)": f"{100 * lo:+.1f} to {100 * hi:+.1f}", "equivalent (±10 pp)": bool(lo > -0.10 and hi < 0.10)})
b1 = pd.DataFrame(rows)
with pd.option_context("display.float_format", "{:.3f}".format):
    print(b1.to_string(index=False))
print("\nMotifs equivalent within ±10 pp, by split:", b1.groupby("split")["equivalent (±10 pp)"].agg(["sum", "size"]).to_dict("index"))

       split            motif  rate  group 1 n  AME (pp)    90% CI (pp)  equivalent (±10 pp)
       onset         loc_mall 0.389        252     4.439   -1.1 to +9.8                 True
       onset   loc_mall_store 0.198        252    -1.132   -5.4 to +3.2                 True
       onset  loc_city_street 0.168        252     0.061   -3.9 to +3.9                 True
       onset        loc_house 0.153        252     8.244  +3.8 to +13.0                False
       onset       loc_school 0.114        252    -1.618   -4.7 to +1.7                 True
       onset        loc_hotel 0.112        252     0.689   -2.5 to +4.1                 True
       onset   loc_restaurant 0.082        252    -3.664   -6.0 to -1.2                 True
       onset        loc_beach 0.080        252     2.662   -0.6 to +5.9                 True
       onset  loc_parking_lot 0.067        252     1.287   -1.3 to +4.0                 True
       onset loc_neighborhood 0.061        252     0.485   -1.9 to +3.

## Findings — registered verdicts and what they mean

| # | Question | Result | Verdict |
|---|---|---|---|
| H1 | Does the person lead the place? | Person → next place +0.117 SD; place → next person −0.003 SD; D = +0.121 (95% CI 0.020–0.213), Holm p = 0.036 | **Hit** |
| H2 | Do people move toward what they treat as good? | −0.245 SD per unit of orientation (−0.436 to −0.055) | **Miss** (registered) |
| H3 | Is a perceived benign presence felt? | +0.353 within dreams (0.221–0.485), Holm p < 0.0001 | **Hit** (not discriminating) |
| H4 | Is approach consistent across a person's dreams? | ICC1 0.148 (−0.056 to 0.353), Holm p = 0.126; 38 authors | **Underdetermined** |
| H5 | Common structure, personal state | Mean ICC1 0.073 for structure, 0.064 for state (Q 0.019, E 0.110) | Descriptive |
| H6 | Does going up help only those in a good state? | Interaction −0.116 (−0.364 to 0.133) | **Underdetermined** |
| B1 | Is the shared structure contagion? | 12–13 of 13 motifs equivalent (±10 pp) across onset, era and first report | Bias check: not contagion |

**H1.**
- **Result.** The person's state carries forward into the quality of the next place. The place's quality does not carry into the person's next state.
- **Robustness.** The direction holds in every variant, with less precision on list-order pairs (+0.061) and with dream fixed effects (+0.065).
- **Interpretation.** This fits the framework (*HH* §§173–176, 192, 547) and runs against an account in which the environment drives the feeling. It does not separate the framework from narrative colouring by the narrator's mood: both are coded from one text.

**H2.** The registered verdict is a miss. The post hoc diagnosis shows the index did not measure what it was meant to:
- 92.7% of negative E·Q products are distress in better-than-neutral places, not comfort in poor ones;
- controlling for the dream's level leaves the estimate unchanged (−0.235, SE 0.149);
- a rank-correlation index gives −0.070 (SE 0.110).

The principle that good is as the experiencer sees it remains untested with this extraction.

**H3.** The person's state is higher where a being he or she perceives as benign is present, robust in every stratum. As a test of influence it establishes the effect. It cannot say what the being is.

**H4 and H6.** Both underdetermined; power is low. H6's point estimate runs against the conditional height prediction.

**Deviation.** 259 self-loop connections were excluded without registration. Including them leaves every conclusion unchanged (H1 D = +0.107, p = 0.010).

**B1.** The shared motifs are as frequent in reports describing childhood or long-standing onset and in 2021–2023 as elsewhere, so community contagion is not their main source. Two exceptions: houses are more frequent in long-standing reports, and malls in an author's first report.